**NOTE:** utility for inspecting specific units — plots the waveform of a chosen unit/spike at a chosen timepoint across nearby channels, styled with a black background (confirmed: `facecolor="k"` on the figure/axes), similar to phy's waveform panel. Edit `unit_mapping`/`unit_indice`/`timepoint_s` etc. to pick what to look at. Standalone, run by hand.

**Output:** a single figure, `waveforms2.png` (filename set by `CFG['fig1_out']`) — saved to the current working directory when the notebook is run, not a fixed path. Visual inspection only, nothing downstream reads this file.

#### IMPORTS

In [2]:
import spikeinterface.full as si
import numpy as np
import matplotlib.pyplot as plt
import os

In [3]:
rec_dir = '/ptmp/gongt/GLC6962_l_g1/preprocessed'
ks4_dir = '/ptmp/gongt/GLC6962_l_g1/KS4/test_MG_params2/sorter_output'
rec = si.load(rec_dir)
sorting = si.read_kilosort(ks4_dir)  # reads from sorter_output/ automatically
print(f"Sampling rate: {rec.get_sampling_frequency()} Hz")
print(f"Units: {sorting.get_unit_ids()[:10]} ...")

Sampling rate: 30000.0 Hz
Units: [ 0  1  2  4  5  6  7 10 13 14] ...


In [4]:
channels = rec.get_num_channels()
fs = rec.get_sampling_frequency()
print(channels)
print(fs)

352
30000.0


In [5]:
print(sorting)

KiloSortSortingExtractor: 340 units - 1 segments - 30.0kHz


In [6]:
unit_mapping = {
    0: 13, 1: 14, 2: 268, 3: 280, 4: 306, 5: 387, 6: 494, 7: 583, 
    8: 650, 9: 710, 10: 714, 11: 721, 12: 740, 13: 742, 14: 753, 
    15: 754, 16: 759, 17: 775, 18: 807, 19: 811, 20: 820, 21: 831, 
    22: 833, 23: 843, 24: 851, 25: 863, 26: 874, 27: 878, 28: 885, 
    29: 892, 30: 897, 31: 903, 32: 908, 33: 914, 34: 919, 35: 936, 
    36: 941, 37: 946, 38: 950
}
unit_indices = sorted(unit_mapping.keys())


In [7]:
# Cell 2: Parameters — set these
unit_indice = 27
unit_id   = unit_mapping[unit_indice]       # the unit you care about
timepoint_s = 0  # timepoint in SECONDS; or use timepoint_samples below
n_ch      = 32       # number of nearest channels
win_ms    = 3.0      # waveform window: ±win_ms/2 around spike

# Find the closest spike to your timepoint
spike_times = sorting.get_unit_spike_train(unit_id, segment_index=0)  # in samples
print(f"Unit {unit_id} has {len(spike_times)} spikes.")
timepoint_samples = int(timepoint_s * fs)

# nearest spike to the requested timepoint
idx_nearest = np.argmin(np.abs(spike_times - timepoint_samples))
center_sample = spike_times[idx_nearest]

print(f"Requested timepoint: {timepoint_s:.4f} s  →  sample {timepoint_samples}")
print(f"Nearest spike:       {center_sample / fs:.4f} s  →  sample {center_sample}")
print(f"Δt = {(center_sample - timepoint_samples)/fs*1000:.2f} ms")


Unit 878 has 415192 spikes.
Requested timepoint: 0.0000 s  →  sample 0
Nearest spike:       0.0963 s  →  sample 2888
Δt = 96.27 ms


In [15]:
import time
import numpy as np
import matplotlib
matplotlib.use("Agg")          # headless
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.collections import LineCollection
from pathlib import Path
import warnings

# ──────────────────────────────────────────────────────────────
#  SpikeInterface Import
# ──────────────────────────────────────────────────────────────
try:
    import spikeinterface.full as si
except ImportError:
    raise ImportError("This script requires spikeinterface. Please install it via 'pip install spikeinterface[full]'")

# ══════════════════════════════════════════════════════════════
#  CONFIG – edit this block to fine-tune everything
# ══════════════════════════════════════════════════════════════
CFG = dict(
    # ── paths ─────────────────────────────────────────────────
    recording_dir   = rec_dir,
    ks_dir          = ks4_dir,

    # ── window ────────────────────────────────────────────────
    start_time_s    = 1568.47,   # seconds from recording start
    window_s        = 500.0,     # seconds to display

    # ── units to inspect ──────────────────────────────────────
    unit_indices    = [15, 16, 33],
    # [1] phy's actual default cluster colors, confirmed from phy/utils/color.py:
    #     cluster 0 -> #0892FC (hardcoded blue), cluster 1 -> #FF0202 (hardcoded red).
    #     cluster 2+ come from colorcet's glasbey_bw_minc_20_minl_30 procedural palette —
    #     the 3rd value below is a placeholder; run the colorcet snippet from chat to get
    #     the exact hex if you want a perfect match.
    unit_color_cycle = ["#0892FC", "#ffbb78", "#FF0202"],

    # ── waveform panel (Figure 1) spacing & layout ────────────
    n_nearest_ch         = 10,     # channels around best channel
    samples_before       = 30,     # ~1 ms before spike sample
    samples_after        = 50,     # ~1.6 ms after spike sample
    waveform_gain         = 0.4,    # amplitude scale of the spike snippets
    fig1_panel_width       = 10,     # inches, PER unit column (n_active columns total)
    fig1_panel_height      = 50.0,   # inches total height — taller = more space between channels
                                      #     (y-axis is real depth in µm, so this directly sets px-per-µm)
    fig1_y_pad             = 30,     # µm of whitespace padded above/below the channel cluster
    max_spikes_per_unit   = 1000,   # cap snippets drawn per unit (None = draw all). Speed lever.
    subsample_seed        = 0,

    # ── trace panel (Figure 2) ─────────────────────────────────
    filter_trace         = False,
    filter_lo            = 300,
    filter_hi            = 6000,
    trace_gain            = 1.0,    # amplitude scale of the continuous traces
    trace_spacing_factor  = 6.0,    # RELATIVE space between channels (multiplies auto percentile spacing)
    fig2_row_spacing       = None,   # ABSOLUTE space between channels in µV — set this to bypass the auto calc entirely
    fig2_figsize           = (20, None),  # (width, height) inches; height=None -> auto from nChan
    dot_size               = 24,
    max_points_per_channel = 8000,  # decimate each trace to this many points for display — big speed lever

    # ── look & feel — black background, bright cycling trace colors ──
    trace_color_cycle = ["#1f77b4", "#d62728", "#ffbb78"],  # only used by Figure 2 (currently disabled)
    trace_alpha     = 0.85,
    trace_lw        = 0.4,
    waveform_alpha_floor = 0.15,
    waveform_alpha_cap   = 0.65,

    # ── output ────────────────────────────────────────────────
    fig1_out        = "waveforms2.png",
    fig2_out        = "traces.png",
    dpi             = 150,
)
# ══════════════════════════════════════════════════════════════

# ──────────────────────────────────────────────────────────────
#  Helpers
# ──────────────────────────────────────────────────────────────

def load_npy(path, name):
    f = Path(path) / name
    if not f.exists():
        return None
    return np.load(f)

def nearest_channels(ch_positions, best_ch_idx, n=32):
    dists = np.linalg.norm(ch_positions - ch_positions[best_ch_idx], axis=1)
    order = np.argsort(dists)
    return order[:n]

def get_cluster_best_channel(cluster_id, spike_clusters, spike_templates,
                              templates, channel_map):
    mask = spike_clusters == cluster_id
    if mask.sum() == 0:
        return 0
    tmpl_indices = spike_templates[mask]
    unique_t, counts_t = np.unique(tmpl_indices, return_counts=True)
    dominant_tmpl = unique_t[np.argmax(counts_t)]
    wf = templates[dominant_tmpl]
    amplitudes = wf.max(axis=0) - wf.min(axis=0)
    best_local = int(np.argmax(amplitudes))
    return best_local

def make_color_palette(n, custom=None):
    """If `custom` hex colors are given, cycle through them exactly.
    Otherwise sample the qualitative colormap by INTEGER index (not a
    normalized float) — sampling tab10 with cmap(i/(n-1)) is the bug that
    produced blue/brown/cyan instead of the expected tab10 sequence."""
    if custom:
        return [custom[i % len(custom)] for i in range(n)]
    cmap = cm.get_cmap("tab20" if n > 10 else "tab10")
    return [cmap(i) for i in range(n)]

def build_snippet_segments(spike_offsets, trace_row, t_axis, pre, post, win_smp, gain, y_base):
    """Build all spike-snippet line segments for ONE channel as a list of
    (x, y) arrays, drawn as a single LineCollection instead of one
    ax.plot() call per spike."""
    segs = []
    for off in spike_offsets:
        if off - pre < 0 or off + post > win_smp:
            continue
        snip = trace_row[off - pre: off + post]
        segs.append(np.column_stack([t_axis, y_base + snip * gain]))
    return segs

# ──────────────────────────────────────────────────────────────
#  Main
# ──────────────────────────────────────────────────────────────

def main(cfg):
    ks_dir = Path(cfg["ks_dir"])
    rec_dir = Path(cfg["recording_dir"])

    print(f"Loading recording from {rec_dir} using SpikeInterface...")
    recording = si.load_extractor(rec_dir)
    sr = recording.get_sampling_frequency()
    si_positions = recording.get_channel_locations()

    start_s   = cfg["start_time_s"]
    win_s     = cfg["window_s"]
    start_smp = int(start_s * sr)
    win_smp   = int(win_s * sr)
    end_smp   = start_smp + win_smp

    print(f"Loading Kilosort outputs from {ks_dir}...")
    spike_times     = load_npy(ks_dir, "spike_times.npy")
    spike_clusters  = load_npy(ks_dir, "spike_clusters.npy")
    spike_templates_arr = load_npy(ks_dir, "spike_templates.npy")
    templates       = load_npy(ks_dir, "templates.npy")
    channel_map     = load_npy(ks_dir, "channel_map.npy")
    ch_pos          = load_npy(ks_dir, "channel_positions.npy")

    pre    = cfg["samples_before"]
    post   = cfg["samples_after"]
    wf_len = pre + post

    unit_indices  = cfg["unit_indices"]
    master_colors = make_color_palette(len(unit_indices), custom=cfg.get("unit_color_cycle"))
    idx_to_color  = {u_idx: master_colors[i] for i, u_idx in enumerate(unit_indices)}

    ks_to_si_idx = []
    for i in range(len(ch_pos)):
        dists = np.linalg.norm(si_positions - ch_pos[i], axis=1)
        ks_to_si_idx.append(np.argmin(dists))
    ks_to_si_idx = np.array(ks_to_si_idx)

    print("Extracting traces...")
    neural_raw = recording.get_traces(start_frame=start_smp, end_frame=end_smp, return_scaled=True)
    neural_raw = neural_raw.T
    nChan = neural_raw.shape[0]

    if cfg["filter_trace"]:
        print("Applying bandpass filtering...")
        from scipy.signal import butter, filtfilt
        nyq = sr / 2
        b, a = butter(3, [cfg["filter_lo"] / nyq, cfg["filter_hi"] / nyq], btype="band")
        neural_filt = filtfilt(b, a, neural_raw, axis=1).astype(np.float32)
    else:
        neural_filt = neural_raw

    in_window = (spike_times > start_smp) & (spike_times < end_smp)
    win_times    = spike_times[in_window]
    win_clusters = spike_clusters[in_window]

    rng = np.random.default_rng(cfg["subsample_seed"])

    # ──────────────────────────────────────────────────────────
    #  PRE-FILTER: Drop empty units
    # ──────────────────────────────────────────────────────────
    active_unit_indices = []
    for u_idx in unit_indices:
        ks_id = unit_mapping[u_idx]
        if np.sum(win_clusters == ks_id) > 0:
            active_unit_indices.append(u_idx)

    n_active = len(active_unit_indices)
    if n_active == 0:
        print("Warning: No spikes found for any specified units in this window! Skipping Figure 1.")
    else:
        # ═════════════════════════════════════════════════════════
        #  FIGURE 1 – Waveform Grid
        # ═════════════════════════════════════════════════════════
        fig1_w = cfg["fig1_panel_width"] * n_active
        fig1_h = cfg["fig1_panel_height"]
        fig1, axes1 = plt.subplots(1, n_active, figsize=(fig1_w, fig1_h), sharey=True, squeeze=False)
        fig1.suptitle("Waveforms Spaced by Probe Depth (Phy Method)", color="w", y=0.99, fontsize=12)
        fig1.patch.set_facecolor("k")

        all_active_y_positions = []
        unit_channel_groups = {}

        for u_idx in active_unit_indices:
            ks_id = unit_mapping[u_idx]
            best_local = get_cluster_best_channel(ks_id, spike_clusters, spike_templates_arr, templates, channel_map)
            nn_idx = nearest_channels(ch_pos, best_local, n=cfg["n_nearest_ch"])
            unit_channel_groups[u_idx] = nn_idx
            all_active_y_positions.extend(ch_pos[nn_idx, 1])

        pad = cfg["fig1_y_pad"]
        min_active_y = min(all_active_y_positions) - pad
        max_active_y = max(all_active_y_positions) + pad

        t_axis = (np.arange(wf_len) - pre) / sr * 1000   # ms

        for i, u_idx in enumerate(active_unit_indices):
            ks_id = unit_mapping[u_idx]
            ax = axes1[0, i]
            ax.set_facecolor("k")
            color_u = idx_to_color[u_idx]

            mask_u = win_clusters == ks_id
            spike_smp_u = win_times[mask_u]

            max_n = cfg.get("max_spikes_per_unit")
            if max_n is not None and len(spike_smp_u) > max_n:
                spike_smp_u = rng.choice(spike_smp_u, size=max_n, replace=False)

            nn_idx = unit_channel_groups[u_idx]

            for ch in nn_idx:
                y_pos = ch_pos[ch, 1]
                ax.axhline(y_pos, color="0.25", lw=0.4, ls="-", alpha=0.3)
                ax.text(t_axis[0], y_pos + 1.5, f"Ch{ch}", color="0.5", fontsize=6.5, va="bottom")

            n_spikes_drawn = len(spike_smp_u)
            alpha = float(np.clip(1.2 / max(n_spikes_drawn, 1) ** 0.3,
                                   cfg["waveform_alpha_floor"], cfg["waveform_alpha_cap"]))
            spike_offsets = (spike_smp_u.astype(int) - start_smp)

            for ch_global_idx in nn_idx:
                si_idx = ks_to_si_idx[ch_global_idx]
                y_pos  = ch_pos[ch_global_idx, 1]
                segs = build_snippet_segments(spike_offsets, neural_filt[si_idx], t_axis,
                                               pre, post, win_smp, cfg["waveform_gain"], y_pos)
                if not segs:
                    continue
                lc = LineCollection(segs, colors=[color_u], alpha=alpha, linewidths=0.5,
                                     rasterized=True)
                ax.add_collection(lc)

            ax.set_title(f"Idx {u_idx}\n[KS {ks_id}]\n({n_spikes_drawn} spks)", color=color_u, fontsize=9)
            ax.tick_params(colors="0.6", labelsize=8)
            ax.set_xlabel("ms", color="0.5", fontsize=7.5)
            ax.set_xlim(t_axis[0], t_axis[-1])
            ax.grid(False)
            for sp_side in ax.spines.values():
                sp_side.set_edgecolor("0.15")

        axes1[0, 0].set_ylabel("Probe Depth (µm)", color="w", fontsize=10)
        axes1[0, 0].set_ylim(min_active_y, max_active_y)

        plt.tight_layout()
        fig1.savefig(cfg["fig1_out"], dpi=cfg["dpi"], facecolor="k")
        print(f"Saved waveform figure: {cfg['fig1_out']}")

    # # ═════════════════════════════════════════════════════════════════════
    # #  FIGURE 2 – continuous trace ordered and labeled by depth
    # #  >>> DELETE FROM HERE TO THE "END FIGURE 2" MARKER BELOW IF YOU DON'T NEED THIS PLOT 
    # # ═════════════════════════════════════════════════════════════════════
    # t0 = time.time()
    # depth_order_idx = np.argsort(ch_pos[:, 1])

    # fig2_w, fig2_h = cfg["fig2_figsize"]
    # if fig2_h is None:
    #     fig2_h = max(8, nChan * 0.18)
    # fig2, ax2 = plt.subplots(figsize=(fig2_w, fig2_h))
    # ax2.set_facecolor("k")
    # fig2.patch.set_facecolor("k")

    # # decimate for display — at dpi=150 and typical figure widths, a trace
    # # can't visually resolve more than a few thousand points/inch anyway, so
    # # plotting millions of raw samples per channel is pure wasted time.
    # max_pts = cfg.get("max_points_per_channel")
    # stride = max(1, win_smp // max_pts) if max_pts else 1
    # t_win = np.linspace(start_s, start_s + win_s, win_smp)[::stride]
    # print(f"  decimation stride = {stride}  ({win_smp} -> {len(t_win)} points/channel)")

    # ch_range = np.percentile(np.abs(neural_filt), 99, axis=1)
    # if cfg.get("fig2_row_spacing") is not None:
    #     trace_spacing = float(cfg["fig2_row_spacing"])          # absolute override
    # else:
    #     trace_spacing = float(np.median(ch_range)) * cfg["trace_spacing_factor"]   # relative
    # if trace_spacing < 1:
    #     trace_spacing = 50.0

    # colors = cfg["trace_color_cycle"]
    # print("Plotting continuous traces ordered and stacked by depth...")
    # for ci in range(nChan):
    #     orig_ks_idx = depth_order_idx[ci]
    #     orig_si_idx = ks_to_si_idx[orig_ks_idx]
    #     offset_ci = ci * trace_spacing
    #     ax2.plot(t_win, neural_filt[orig_si_idx, ::stride] * cfg["trace_gain"] + offset_ci,
    #               color=colors[ci % len(colors)], lw=cfg["trace_lw"], alpha=cfg["trace_alpha"],
    #               rasterized=True)
    #     if (ci + 1) % 50 == 0 or ci == nChan - 1:
    #         print(f"    channel {ci+1}/{nChan}  ({time.time()-t0:.1f}s elapsed)")

    # print("Overlaying verified spike dots at true channel locations…")
    # for u_idx in unit_indices:
    #     ks_id = unit_mapping[u_idx]
    #     color_u = idx_to_color[u_idx]

    #     mask_u = win_clusters == ks_id
    #     spike_smp_u = win_times[mask_u]
    #     if len(spike_smp_u) == 0:
    #         continue

    #     if templates is not None and spike_templates_arr is not None:
    #         best_local_ks = get_cluster_best_channel(ks_id, spike_clusters, spike_templates_arr, templates, channel_map)
    #     else:
    #         best_local_ks = 0

    #     depth_ordered_idx = np.where(depth_order_idx == best_local_ks)[0][0]
    #     y_dot_base = depth_ordered_idx * trace_spacing
    #     orig_si_idx = ks_to_si_idx[best_local_ks]

    #     spike_offsets = (spike_smp_u - start_smp).astype(int)
    #     valid_masks = (spike_offsets >= 0) & (spike_offsets < win_smp)
    #     spike_offsets = spike_offsets[valid_masks]

    #     if len(spike_offsets) > 0:
    #         t_sp = start_s + spike_offsets / sr
    #         y_val = neural_filt[orig_si_idx, spike_offsets] * cfg["trace_gain"] + y_dot_base   # full-res, not decimated

    #         ax2.scatter(t_sp, y_val, color=color_u, s=cfg["dot_size"], zorder=5,
    #                     edgecolors='w', linewidths=0.3)
    #         first_idx = np.argmin(t_sp)
    #         ax2.text(t_sp[first_idx] + 0.004, y_val[first_idx], f"Idx {u_idx}",
    #                   color=color_u, fontsize=8, weight='bold', va='center',
    #                   bbox=dict(facecolor='black', alpha=0.7, edgecolor='none', pad=1), zorder=6)

    # legend_handles = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=idx_to_color[u_idx],
    #                               markersize=6, label=f"Idx {u_idx} (KS {unit_mapping[u_idx]})", lw=0)
    #                    for u_idx in unit_indices]
    # ax2.legend(handles=legend_handles, loc="upper right", framealpha=0.5, fontsize=8,
    #             labelcolor="w", facecolor="0.1")

    # tick_step = max(1, nChan // 30)
    # yticks = [ci * trace_spacing for ci in range(0, nChan, tick_step)]
    # ytick_labels = [f"{ch_pos[depth_order_idx[ci], 1]} µm (Ch {depth_order_idx[ci]})" for ci in range(0, nChan, tick_step)]

    # ax2.set_yticks(yticks)
    # ax2.set_yticklabels(ytick_labels, fontsize=7, color="w")
    # ax2.set_xlabel("Time (s)", color="w", fontsize=9)
    # ax2.set_ylabel("Physical Depth along Probe Channel Map", color="w", fontsize=9)
    # ax2.set_xlim(start_s, start_s + win_s)
    # ax2.set_title(f"Continuous Traces (Labeled by True Depth) | t = {start_s:.2f}–{start_s+win_s:.2f} s",
    #                color="w", fontsize=10)
    # ax2.tick_params(colors="w")
    # for sp_side in ax2.spines.values():
    #     sp_side.set_edgecolor("0.3")

    # plt.tight_layout()
    # fig2.savefig(cfg["fig2_out"], dpi=cfg["dpi"], facecolor="k")
    # print(f"Saved traces figure: {cfg['fig2_out']}  (total {time.time()-t0:.1f}s)")
    # # ═════════════════════════════════════ END FIGURE 2 ═════════════════════════════════════

    print("Done.")

if __name__ == "__main__":
    main(CFG)

Loading recording from /ptmp/gongt/GLC6962_l_g1/preprocessed using SpikeInterface...
Loading Kilosort outputs from /ptmp/gongt/GLC6962_l_g1/KS4/test_MG_params2/sorter_output...


/tmp/ipykernel_113821/3822656512.py:136: DeprecationWarning: load_extractor() is deprecated and will be removed in version 0.104.0. Please use load() instead.
  recording = si.load_extractor(rec_dir)


Extracting traces...


/tmp/ipykernel_113821/3822656512.py:169: DeprecationWarning: `return_scaled` is deprecated and will be removed in version 0.105.0. Use `return_in_uV` instead.
  neural_raw = recording.get_traces(start_frame=start_smp, end_frame=end_smp, return_scaled=True)


Saved waveform figure: waveforms2.png
Done.
